In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from enum import Enum
from copy import copy
from scipy.stats import norm
import pandas as pd
import numpy as np

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utils.core_functions import * 

from _utils.portfoliohandcrafiting import *
from _utils.strategies.trend_simple_filter import *



In [9]:
DATA_DIR = '../_databases'
df_win = pd.read_excel((DATA_DIR + '/WINFUT.xlsx'),decimal=',',index_col='date')
df_win.sort_index(inplace=True)

df_wdo = pd.read_excel((DATA_DIR + '/WDOFUT.xlsx'),decimal=',',index_col='date')
df_wdo.sort_index(inplace=True)

In [15]:
adjusted_price = df_wdo['adjusted_close']
current_price = df_wdo['close']
multiplier = 10
risk_target_tau = 0.4
fx_series = pd.Series(1, index=df_wdo.index)  ## FX rate

capital = 100000


instrument_risk = standardDeviation(
    adjusted_price=adjusted_price,
    current_price=current_price,
    use_perc_returns=True,
    annualise_stdev=True,
)


In [ ]:
br_adjusted_prices, current_prices = get_data_dict(['winfut','wdofut'])
multipliers = dict(winfut=0.2, wdofut=10)
risk_target_tau = 0.4

fx_series_dict = create_fx_series_given_adjusted_prices_dict(br_adjusted_prices,dict(winfut=1, wdofut=1))

capital = 1000000
idm = 1.5
instrument_weights = dict(winfut=0.5, wdofut=0.5)

std_dev_dict = calculate_variable_standard_deviation_for_risk_targeting_from_dict(
adjusted_prices=br_adjusted_prices,
current_prices=current_prices,
annualise_stdev=True,  ## can also be False if want to use daily price diff
use_perc_returns=True,  ## can also be False if want to use daily price diff
)


# 50% - 50%  US portfolio with EWMAC long only filter
br_average_position_contracts_dict = (
        calculate_position_series_given_variable_risk_for_dict(
            capital=capital,
            risk_target_tau=risk_target_tau,
            idm=idm,
            weights=instrument_weights,
            std_dev_dict=std_dev_dict,
            fx_series_dict=fx_series_dict,
            multipliers=multipliers,
        )
    )

In [16]:
position_contracts_held = calculate_position_series_given_variable_risk(
    capital=capital,
    fx=fx_series,
    instrument_risk=instrument_risk,
    risk_target_tau=risk_target_tau,
    multiplier=multiplier,
)


In [17]:
perc_return = calculate_perc_returns(
       position_contracts_held=position_contracts_held,
       adjusted_price=adjusted_price,
       fx_series=fx_series,
       capital_required=capital,
       multiplier=multiplier,
   )

In [18]:
# --- Plot retorno acumulado
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=perc_return.index,
        y=perc_return.cumsum()*100,
        name="WINFUT - Estratégia com ajuste de risco",
        line=dict(color="blue"),
    )
)

fig.update_layout(
    title=f"Estratégia com Ajuste de Risco - alvo std {risk_target_tau*100}%- Retorno Acumulado",
    xaxis_title="Data",
    yaxis_title="Retorno Acumulado (%)",
    template="plotly_white",
    showlegend=True,
    legend=dict(
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1),
    height=600,
    width=1000,
)

fig.show()

# --- Exibir estatísticas
print("Estatísticas de Desempenho - WIN")
print("Estatísticas na Frequência Diária")
print(calculate_stats(perc_return))
print("================================")
print("Estatísticas na Frequência Mensal")
print(calculate_stats(perc_return, freq=MONTH))
print("================================")
print("Capital Mínimo Necessário para 50 Contratos de WIN")
print(
    calculate_minimum_capital(
        multiplier=multiplier,
        target_risk=risk_target_tau,
        fx=1,
        ann_volatility=instrument_risk[-1],
        price=current_price.iloc[-1],
        n_contracts=50,
    )
)

print("Turnover WIN")
print(
        calculate_turnover(
            position_contracts_held, average_position=position_contracts_held
        )
    )

Estatísticas de Desempenho - WIN
Estatísticas na Frequência Diária
{'ann_mean': -0.0335551376682145, 'ann_std': 0.3648735794194085, 'sharpe': -0.09196373637578217, 'skew': 0.4059096353070475, 'avg_drawdown': 1.1088252358630557, 'max_drawdown': 2.381064909015138, 'quant_lower': 1.03752420212038, 'quant_upper': 1.3397011836403354}
Estatísticas na Frequência Mensal
{'ann_mean': -0.6911539941660284, 'ann_std': 1.6810222906390377, 'sharpe': -0.41115099901696567, 'skew': 0.6730451569108672, 'avg_drawdown': 0.9993571866152808, 'max_drawdown': 2.2312230217441074, 'quant_lower': 0.817870966306303, 'quant_upper': 1.6781210885420998}
Capital Mínimo Necessário para 50 Contratos de WIN
931626.5811205091
Turnover WIN
5.201194257805801


In [19]:
position_contracts_held

date
2005-05-17         NaN
2005-05-18         NaN
2005-05-19    9.602545
2005-05-20    6.373547
2005-05-23    7.304530
                ...   
2025-10-27    5.108784
2025-10-28    5.205603
2025-10-29    5.303823
2025-10-30    5.301876
2025-10-31    5.366957
Length: 5070, dtype: float64